# Realce no Domínio da Frequência e Filtros no domínio da Frequência

In [ ]:
# Download do OpenCV
!pip install opencv-python

In [ ]:
# Importação das bibliotecas
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow

In [ ]:
# Download do repositório
if os.path.isdir("pdsi"):
  print("Diretório já existe!")
else:
  !git clone https://github.com/NaldoHoracio/pdsi

# Funções auxiliares

In [ ]:
# Silencia os avisos de metadados dos arquivos .tif
cv2.utils.logging.setLogLevel(cv2.utils.logging.LOG_LEVEL_ERROR)

def ler_cinza(nome):
    # Lê a imagem do repositório em tons de cinza
    return cv2.imread(f'pdsi/imagens/{nome}', cv2.IMREAD_GRAYSCALE)

def espectro(img):
    # DFT 2D centrada: a frequência (0,0) vai para o centro da matriz
    return np.fft.fftshift(np.fft.fft2(img.astype(np.float64)))

def log_espectro(F):
    # Mapeamento logarítmico para visualizar o espectro: log(1 + |F|)
    return np.log1p(np.abs(F))

def distancia(shape):
    # Matriz D(u,v): distância de cada ponto ao centro do plano de frequências
    M, N = shape
    u, v = np.meshgrid(np.arange(M) - M / 2, np.arange(N) - N / 2, indexing='ij')
    return np.sqrt(u**2 + v**2)

def filtrar(img, H):
    # Pipeline da filtragem: F = DFT(f) centrada -> G = H·F -> f' = Re(IDFT(G))
    G = H * espectro(img)
    return np.real(np.fft.ifft2(np.fft.ifftshift(G)))

def normalizar(img):
    # Reescala qualquer intervalo de valores para [0, 255] (uint8) para exibição
    return cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

def mostrar(imagens, titulos, colunas=4, tamanho=4):
    # Exibe uma lista de imagens lado a lado
    linhas = int(np.ceil(len(imagens) / colunas))
    plt.figure(figsize=(tamanho * colunas, tamanho * linhas))
    for k, (im, t) in enumerate(zip(imagens, titulos)):
        plt.subplot(linhas, colunas, k + 1); plt.imshow(im, cmap='gray'); plt.title(t, fontsize=10); plt.axis('off')
    plt.tight_layout(); plt.show()

# Transformada Discreta de Fourier (DFT) 2D

O espectro mostra **quais frequências** compõem a imagem: o centro concentra as baixas frequências (regiões homogêneas) e as bordas do espectro, as altas (bordas, detalhes, ruído).

In [ ]:
img = ler_cinza('cameraman.png')
img.shape

In [ ]:
F = espectro(img)
espectro_linear = np.abs(F)       # magnitude |F(u,v)| (espectro de Fourier)
fase = np.angle(F)                # fase φ(u,v)

mostrar([img, normalizar(espectro_linear), log_espectro(F), fase],
        ['Imagem original', 'Espectro (mapeamento linear)', 'Espectro (mapeamento log)', 'Fase'])

## Centralização com $(-1)^{x+y}$

Os slides centralizam a transformada multiplicando a imagem por $(-1)^{x+y}$ antes da DFT. O resultado é equivalente ao `np.fft.fftshift`.

In [ ]:
M, N = img.shape
x, y = np.meshgrid(np.arange(M), np.arange(N), indexing='ij')
sinal = (-1.0) ** (x + y)                       # matriz alternando +1 e -1

F_slides = np.fft.fft2(img * sinal)             # centralização "à moda dos slides"
print('Diferença máxima entre os dois métodos:', np.abs(np.abs(F_slides) - np.abs(F)).max().round(6))
print('Valor médio da imagem:', img.mean().round(2), '| F(0,0)/(M·N):', (np.abs(F[M//2, N//2]) / (M * N)).round(2))

## Reconstrução sem perda (transformada inversa)

In [ ]:
reconstruida = np.real(np.fft.ifft2(np.fft.ifftshift(F)))   # IDFT do espectro sem nenhum filtro
print('Erro máximo da reconstrução:', np.abs(reconstruida - img).max())
mostrar([img, normalizar(reconstruida)], ['Original', 'IDFT(DFT(f))'], colunas=2)

# Filtro básico: *notch* em F(0,0)

Zerar o termo F(0,0) remove o **valor médio** da imagem: o resultado tem média zero.

In [ ]:
H_notch = np.ones(img.shape)
H_notch[M // 2, N // 2] = 0                     # zera apenas a componente DC (centro do espectro)

g_notch = filtrar(img, H_notch)
print('Média antes:', img.mean().round(2), '| média depois:', g_notch.mean().round(6))
mostrar([img, normalizar(g_notch)], ['Original', 'Notch F(0,0)=0 (reescalada)'], colunas=2)

# Filtros passa-baixa (suavização)

| Filtro | H(u,v) |
|---|---|
| Ideal (ILPF) | 1 se D(u,v) ≤ D₀; 0 caso contrário |
| Butterworth (BLPF) | 1 / [1 + (D/D₀)^{2n}] |
| Gaussiano (GLPF) | exp(−D² / 2D₀²) |

In [ ]:
def passa_baixa_ideal(shape, D0):
    return (distancia(shape) <= D0).astype(np.float64)              # corte abrupto em D0

def passa_baixa_butterworth(shape, D0, n=2):
    return 1 / (1 + (distancia(shape) / D0) ** (2 * n))             # transição suave controlada pela ordem n

def passa_baixa_gaussiano(shape, D0):
    return np.exp(-(distancia(shape) ** 2) / (2 * D0 ** 2))         # transição suave, sem ondulações

## Visualizando H(u,v) em 2D e o perfil 1D

In [ ]:
D0 = 40
filtros = {'Ideal': passa_baixa_ideal(img.shape, D0),
           'Butterworth n=2': passa_baixa_butterworth(img.shape, D0, 2),
           'Gaussiano': passa_baixa_gaussiano(img.shape, D0)}

mostrar(list(filtros.values()), [f'{k} (D0={D0})' for k in filtros], colunas=3)

plt.figure(figsize=(7, 3))
for nome, H in filtros.items():
    plt.plot(H[M // 2, N // 2:N // 2 + 120], label=nome)               # perfil do centro para a borda
plt.axvline(D0, ls='--', c='gray'); plt.xlabel('D(u,v)'); plt.ylabel('H(u,v)'); plt.legend(); plt.title('Perfis dos filtros passa-baixa')
plt.show()

## Potência retida pelo círculo de raio D₀

Uma forma de escolher D₀ é medir a porcentagem β da potência total $P_T = \sum |F(u,v)|^2$ que fica dentro do círculo.

In [ ]:
potencia = np.abs(F) ** 2
D = distancia(img.shape)
for raio in [10, 30, 60, 160, 460]:
    beta = 100 * potencia[D <= raio].sum() / potencia.sum()   # β = 100 · P(dentro do círculo) / P_T
    print(f'D0 = {raio:3d} -> β = {beta:.2f}% da potência')

## Passa-baixa ideal: efeito *ringing*

O corte abrupto gera ondulações (anéis) em torno das bordas.

In [ ]:
raios = [10, 30, 60, 160]
resultados = [filtrar(img, passa_baixa_ideal(img.shape, r)) for r in raios]
mostrar([img] + [normalizar(r) for r in resultados], ['Original'] + [f'ILPF D0={r}' for r in raios], colunas=5)

## Ideal × Butterworth × Gaussiano (mesmo D₀)

In [ ]:
D0 = 30
mostrar([img] + [normalizar(filtrar(img, H)) for H in
                 [passa_baixa_ideal(img.shape, D0), passa_baixa_butterworth(img.shape, D0, 2), passa_baixa_gaussiano(img.shape, D0)]],
        ['Original', 'Ideal', 'Butterworth n=2', 'Gaussiano'])

## Butterworth: efeito da ordem n

Quanto maior n, mais o Butterworth se aproxima do filtro ideal (e o *ringing* volta a aparecer).

In [ ]:
texto = ler_cinza('text.tif')
ordens = [1, 2, 5, 20]
mostrar([texto] + [normalizar(filtrar(texto, passa_baixa_butterworth(texto.shape, 60, n))) for n in ordens],
        ['Original'] + [f'BLPF D0=60, n={n}' for n in ordens], colunas=5)

## Aplicação do Gaussiano: unir caracteres falhados

Como nos slides, a suavização preenche pequenas falhas nos caracteres de um texto de baixa resolução.

In [ ]:
mostrar([texto] + [normalizar(filtrar(texto, passa_baixa_gaussiano(texto.shape, d))) for d in [100, 80]],
        ['Original', 'GLPF D0=100', 'GLPF D0=80'], colunas=3)

# Filtros passa-alta (aguçamento)

Obtidos a partir dos passa-baixa: $H_{pa}(u,v) = 1 - H_{pb}(u,v)$.

In [ ]:
def passa_alta_ideal(shape, D0):
    return 1 - passa_baixa_ideal(shape, D0)

def passa_alta_butterworth(shape, D0, n=2):
    return 1 - passa_baixa_butterworth(shape, D0, n)

def passa_alta_gaussiano(shape, D0):
    return 1 - passa_baixa_gaussiano(shape, D0)

In [ ]:
D0 = 30
Hs = [passa_alta_ideal(img.shape, D0), passa_alta_butterworth(img.shape, D0, 2), passa_alta_gaussiano(img.shape, D0)]
mostrar(Hs, ['H ideal', 'H Butterworth n=2', 'H gaussiano'], colunas=3)
# A saída do passa-alta tem média ~0 (perde F(0,0)), por isso é reescalada para exibir
mostrar([img] + [normalizar(filtrar(img, H)) for H in Hs], ['Original', 'IHPF', 'BHPF', 'GHPF'])

## Passa-alta + constante

Somar uma constante ao filtro devolve parte das baixas frequências (o "fundo" da imagem): $H = k + H_{pa}$.

In [ ]:
H_k = 0.5 + passa_alta_butterworth(img.shape, 30, 2)   # k = 0.5 mantém metade das baixas frequências
mostrar([img, normalizar(filtrar(img, passa_alta_butterworth(img.shape, 30, 2))), normalizar(filtrar(img, H_k))],
        ['Original', 'BHPF', 'BHPF + 0.5'], colunas=3)

# Filtros rejeita-faixa e passa-faixa

Atuam sobre um **anel** de frequências de centro D₀ e largura W, e são úteis na restauração (remoção de ruído periódico).

In [ ]:
def rejeita_faixa_gaussiano(shape, D0, W):
    D = distancia(shape) + 1e-8                                     # evita divisão por zero no centro
    return 1 - np.exp(-((D ** 2 - D0 ** 2) / (D * W)) ** 2)         # zera o anel em torno de D0

def passa_faixa_gaussiano(shape, D0, W):
    return 1 - rejeita_faixa_gaussiano(shape, D0, W)                # mantém só o anel

In [ ]:
# Ruído periódico sintético: uma senoide somada à imagem gera picos simétricos no espectro
ruido = 40 * np.sin(2 * np.pi * (x * 60 / M + y * 60 / N))
img_ruido = np.clip(img + ruido, 0, 255)
raio_ruido = np.hypot(60, 60)                                       # distância dos picos ao centro (~85)

H_rf = rejeita_faixa_gaussiano(img.shape, raio_ruido, 10)
restaurada = filtrar(img_ruido, H_rf)
mostrar([img_ruido, log_espectro(espectro(img_ruido)), H_rf, np.clip(restaurada, 0, 255)],
        ['Com ruído periódico', 'Espectro (picos do ruído)', 'Rejeita-faixa gaussiano', 'Restaurada'])
mostrar([passa_faixa_gaussiano(img.shape, raio_ruido, 10), normalizar(filtrar(img_ruido, passa_faixa_gaussiano(img.shape, raio_ruido, 10)))],
        ['Passa-faixa gaussiano', 'Ruído isolado pelo passa-faixa'], colunas=2)

# Laplaciano no domínio da frequência

$H(u,v) = -4\pi^2 D^2(u,v)$ e a imagem realçada é $g = f - \nabla^2 f$ (máscara com centro negativo, c = −1).

In [ ]:
lua = ler_cinza('blurry_moon.tif').astype(np.float64) / 255           # normaliza f para [0, 1]

P, Q = lua.shape
D2 = distancia(lua.shape) ** 2 / (P * Q)                               # D² em frequência normalizada (ciclos/pixel)
H_lap = -4 * np.pi ** 2 * D2

lap = filtrar(lua, H_lap)
lap = lap / np.abs(lap).max()                                          # escala o Laplaciano para [-1, 1]
g_lap = np.clip(lua - lap, 0, 1)                                       # g = f - ∇²f

mostrar([lua, normalizar(lap), g_lap], ['Original', 'Laplaciano (reescalado)', 'Realçada: f − ∇²f'], colunas=3)

# Alto reforço e ênfase de altas frequências

- **Alto reforço (*highboost*)**: $H_{ar} = (A - 1) + H_{pa}$, com A ≥ 1.
- **Ênfase de altas frequências**: $H_{eaf} = a + b\,H_{pa}$, com a ≥ 0 e b > a. Costuma ser seguida de equalização de histograma.

In [ ]:
def alto_reforco(img, H_pa, A):
    return filtrar(img, (A - 1) + H_pa)                                 # A = 1 equivale ao passa-alta puro

H_pa = passa_alta_gaussiano(lua.shape, 30)
resultados = [np.clip(alto_reforco(lua, H_pa, A), 0, 1) for A in [1.7, 2.0, 2.7]]
mostrar([lua] + resultados, ['Original', 'A = 1.7', 'A = 2.0', 'A = 2.7'])

In [ ]:
raiox = ler_cinza('xray.png').astype(np.float64)

H_bhp = passa_alta_butterworth(raiox.shape, 40, 2)
g_hp = filtrar(raiox, H_bhp)                                           # passa-alta puro: perde o fundo
g_eaf = filtrar(raiox, 0.5 + 2.0 * H_bhp)                              # ênfase: a = 0.5, b = 2.0
g_eq = cv2.equalizeHist(normalizar(g_eaf))                             # equalização realça o contraste final

mostrar([raiox, normalizar(g_hp), normalizar(g_eaf), g_eq],
        ['(a) Raio-X original', '(b) Passa-alta Butterworth', '(c) Ênfase de altas freq.', '(d) (c) + equalização'])

# Filtragem homomórfica

Modelo $f(x,y) = i(x,y)\,r(x,y)$: a iluminância **i** varia devagar (baixa frequência) e a refletância **r** varia rápido (alta frequência).

Etapas: $\ln f \rightarrow$ DFT $\rightarrow H \rightarrow$ IDFT $\rightarrow \exp$, com
$H(u,v) = (\gamma_H - \gamma_L)\left[1 - e^{-c\,D^2/D_0^2}\right] + \gamma_L$, onde $\gamma_L < 1$ atenua a iluminação e $\gamma_H > 1$ reforça os detalhes.

In [ ]:
def homomorfico(img, D0=30, gamaL=0.5, gamaH=1.5, c=1.0):
    z = np.log1p(img.astype(np.float64))                               # ln transforma o produto i·r em soma
    D2 = distancia(img.shape) ** 2
    H = (gamaH - gamaL) * (1 - np.exp(-c * D2 / D0 ** 2)) + gamaL      # passa-alta gaussiano modificado
    s = filtrar(z, H)
    return np.expm1(s), H                                              # exp desfaz o logaritmo

def normalizar_percentil(img, p_min=1, p_max=99):
    # Reescala ignorando os 1% de valores extremos, que o exp tende a amplificar
    lo, hi = np.percentile(img, (p_min, p_max))
    return (np.clip((img - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)

coluna = ler_cinza('spine.jpg')
g_homo, H_homo = homomorfico(coluna)

plt.figure(figsize=(7, 3))
plt.plot(H_homo[coluna.shape[0] // 2, coluna.shape[1] // 2:]); plt.xlabel('D(u,v)'); plt.ylabel('H(u,v)')
plt.title('Perfil do filtro homomórfico (γL=0.5, γH=1.5)'); plt.show()

mostrar([coluna, normalizar(g_homo), normalizar_percentil(g_homo)],
        ['Original (iluminação irregular)', 'Homomórfica (min-max)', 'Homomórfica (percentis 1–99)'], colunas=3)

# Referências

- GONZALEZ, R. C.; WOODS, R. E. *Processamento de imagens digitais*. Capítulo 4 – Filtragem no domínio da frequência.
- Documentação NumPy FFT: https://numpy.org/doc/stable/reference/routines.fft.html
- Tutorial OpenCV de Transformada de Fourier: https://docs.opencv.org/4.x/de/dbc/tutorial_py_fourier_transform.html